# LangChain Using Tools

In [1]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [3]:
model = "gpt-5.6-sol"

In [4]:
llm = ChatOpenAI(
    model=model,
    temperature=0.0,
    max_tokens=500,
    timeout=500,
    reasoning_effort="low",
    max_retries=2,
)

In [5]:
class ToolArgsInput(BaseModel):
    a: int = Field(..., description="The first number to add")
    b: int = Field(..., description="The second number to add")

In [6]:
@tool("add", args_schema=ToolArgsInput, description="Add two numbers")
def add(a: int, b: int) -> int:
    """Add two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    """
    return a + b

In [7]:
@tool("multiply", args_schema=ToolArgsInput, description="Multiply two numbers")
def multiply(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    """
    return a * b

In [8]:
tools = [add, multiply]
tool_map = {tool.name: tool for tool in tools}

In [9]:
tool_map

{'add': StructuredTool(name='add', description='Add two numbers', args_schema=<class '__main__.ToolArgsInput'>, func=<function add at 0x0000020D5CDE5900>),
 'multiply': StructuredTool(name='multiply', description='Multiply two numbers', args_schema=<class '__main__.ToolArgsInput'>, func=<function multiply at 0x0000020D5CDE59B0>)}

In [10]:
print(tool_map)

{'add': StructuredTool(name='add', description='Add two numbers', args_schema=<class '__main__.ToolArgsInput'>, func=<function add at 0x0000020D5CDE5900>), 'multiply': StructuredTool(name='multiply', description='Multiply two numbers', args_schema=<class '__main__.ToolArgsInput'>, func=<function multiply at 0x0000020D5CDE59B0>)}


In [11]:
class OutputSchema(BaseModel):
    """Schema for response."""

    answer: str
    justification: str

In [12]:
llm_with_tools = llm.bind_tools(tools, response_format=OutputSchema, strict=True)

In [13]:
question = "What is the sum of 5 and 10, and then multiply the result by 2?"

In [14]:
messages = [
    SystemMessage(
        """
            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        """
    ),
    HumanMessage(question),
]

In [15]:
ai_message = llm_with_tools.invoke(messages)

In [16]:
ai_message

AIMessage(content=[{'id': 'rs_0d7ad83d176ba34a006ab3d8c050d087d2b34f9a1afdd2cbc0', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs9jB5nwimtq1KbwK47Q9SWcJtz5zUCgUKlTmzf-DhaCtNVGljJDPTNIw5n9Zd8RBmZWUOpXD9X_HGXj3ykt-Mu4625nuVpBu5Yvk6zDrJboc7eruGPiv60gQhTCkj9fPg2-0HuFT4UCOUY1myzW6qIiWJBVGMY3tuuZQyX7LQmTWwchev9D7wkdUy5WhqKGBbX5Jx3flrhMS7mmRIHnpntsKAngTyNfXWfzH5xOg_rO7K91QXfbeQWwOd88dGGR_qM5Hmu9jozj0l2pOTd8fX4eUwHbNcPgQN8JvZDII0hmqjGBuF4BYQ1uRm8vq8MZThQnxkdqlj2R9QYHY52x5jI8cye95XpB0862kJ5GYvW4hUBMM1v7lwoltPHe1SGTMewBr8itjGctLe2A9egIpOEjdss24oA3uLNOVzbfaQ2FONjN1XvaXBbHAVLK-K0GozjnT65kTgms7l2SM0wB3L81EBYw9945Pp5ZstMi0amuRucWwvJ32O5ydeWYjQrfO17cPzDrxYG4PGBU-P8Kuhrkcc_jf82YC2d7P__C7_nhKUcSuJAdahzDI-qozkbTdfFBevB5ut7LUP4FebY972jbbsWNxZyZTyt-Ndbij6B15gFmy2rTmBXuNa4vd_MZ0atAXO11Ol1lqnLIjxwf2fLzz7lMDBzFuVb78EA1QXhZoQaDcTqHZumPAkFP8ZPYR5cUaF3dtsxliqgJ7S2rCDozWlLS6m4YIOuWQv_h-3z9os81UfsKhPuGhthYYx_et8hQcixrMpp8cUqsA3YqhC55S8wqtKf3_OxJDzSU8RoHsZbfndFFCdITGEe_6eSoZfCFa

In [17]:
ai_message.tool_calls

[{'name': 'add',
  'args': {'a': 5, 'b': 10},
  'id': 'call_yoM4owueEBvYBp9TRwCnUdST',
  'type': 'tool_call'}]

In [18]:
messages.append(ai_message)

In [19]:
messages

[SystemMessage(content="\n            You're a helpful assistant that can perform mathematical operations using the provided tools only.\n            You can only use the tools provided and cannot perform any calculations on your own.\n            If you need to perform a calculation, you must use the appropriate tool.\n            You can use the following tools:\n            1. add: Adds two numbers together.\n            2. multiply: Multiplies two numbers together.\n            When you need to use a tool, you must call it in the following format:\n            tool_name(arg1, arg2)\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the sum of 5 and 10, and then multiply the result by 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'id': 'rs_0d7ad83d176ba34a006ab3d8c050d087d2b34f9a1afdd2cbc0', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs9jB5nwimtq1KbwK47Q9SWcJtz5zUCgUKlTmzf-DhaCtNVGljJD

In [20]:
for message in messages:
    message.pretty_print()

================================ System Message ================================


            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        
================================ Human Message =================================

What is the sum of 5 and 10, and then multiply the result by 2?
================================== Ai Message ==================================

[{'id': 'rs_0d7ad83d176ba34a006ab3d8c050d087d2b34f9a1afdd2cbc0', 'summary': [], 'type': 'reasoning', 'content': [], 'enc

In [22]:
parsed_tool_calls = ai_message.tool_calls

In [23]:
parsed_tool_calls

[{'name': 'add',
  'args': {'a': 5, 'b': 10},
  'id': 'call_yoM4owueEBvYBp9TRwCnUdST',
  'type': 'tool_call'}]

In [24]:
for tool_call in parsed_tool_calls:
    tool_call_id = tool_call["id"]
    function_name = tool_call["name"]
    arguments = tool_call["args"]
    func = tool_map[function_name]
    result = func.invoke(arguments)
    tool_message = ToolMessage(
        content=result,
        name=function_name,
        tool_call_id=tool_call_id,
    )
    messages.append(tool_message)

In [25]:
messages

[SystemMessage(content="\n            You're a helpful assistant that can perform mathematical operations using the provided tools only.\n            You can only use the tools provided and cannot perform any calculations on your own.\n            If you need to perform a calculation, you must use the appropriate tool.\n            You can use the following tools:\n            1. add: Adds two numbers together.\n            2. multiply: Multiplies two numbers together.\n            When you need to use a tool, you must call it in the following format:\n            tool_name(arg1, arg2)\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the sum of 5 and 10, and then multiply the result by 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'id': 'rs_0d7ad83d176ba34a006ab3d8c050d087d2b34f9a1afdd2cbc0', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs9jB5nwimtq1KbwK47Q9SWcJtz5zUCgUKlTmzf-DhaCtNVGljJD

In [26]:
for message in messages:
    message.pretty_print()

================================ System Message ================================


            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        
================================ Human Message =================================

What is the sum of 5 and 10, and then multiply the result by 2?
================================== Ai Message ==================================

[{'id': 'rs_0d7ad83d176ba34a006ab3d8c050d087d2b34f9a1afdd2cbc0', 'summary': [], 'type': 'reasoning', 'content': [], 'enc